# HLVid NVILA-HD Input / Output Report Notebook

이 노트북은 `scripts/evaluate_hlvid_nvila.py`로 HLVid를 돌릴 때 AutoGaze, SigLIP, NVILA가 어떤 입력을 받고 어떤 출력을 만드는지 확인하는 용도입니다.

핵심: HLVid 재현 경로는 `infer_full.py + resize_then_chop`가 아니라 `evaluate_hlvid_nvila.py + official NVILA-HD processor`입니다.

## 1. 실행 명령

모델을 로드하지 않는 dry-run:

In [ ]:
!python scripts/evaluate_hlvid_nvila.py \
  --config configs/poc_inference/hlvid_nvila_hd_smoke.yaml \
  --dataset-path /path/to/hlvid_sample.jsonl \
  --video-root /path/to/videos \
  --output-dir outputs/hlvid_nvila_dry_run \
  --dry-run

로컬 weight로 subset 실행:

In [ ]:
!python scripts/evaluate_hlvid_nvila.py \
  --config configs/poc_inference/hlvid_nvila_hd_subset.yaml \
  --dataset-name bfshi/HLVid \
  --model-path weights/NVILA-8B-HD-Video \
  --processor-path weights/NVILA-8B-HD-Video \
  --allow-real-model-loading \
  --local-files-only \
  --max-samples 20 \
  --output-dir outputs/hlvid_nvila_real

AutoGaze ON/OFF를 같은 데이터로 비교:

In [ ]:
!python scripts/evaluate_hlvid_nvila.py \
  --config configs/poc_inference/hlvid_nvila_hd_subset.yaml \
  --dataset-path /data/HLVid/annotations.jsonl \
  --video-root /data/HLVid/videos \
  --model-path weights/NVILA-8B-HD-Video \
  --processor-path weights/NVILA-8B-HD-Video \
  --allow-real-model-loading \
  --local-files-only \
  --max-samples 20 \
  --compare-autogaze-on-off \
  --output-dir outputs/hlvid_nvila_on_off

로컬 데이터셋에서 `--dataset-path`는 비디오 폴더가 아니라 annotation 파일 또는 annotation 파일이 들어 있는 디렉터리입니다. 상대 `video_path` 앞에는 `--video-root`가 붙습니다. 예: `video_path=clip.mp4`, `video_root=/data/HLVid/videos`이면 실제 입력은 `/data/HLVid/videos/clip.mp4`입니다.

## 2. Official High-Resolution Path

NVILA-HD processor가 실제로 하는 일:

```text
source video path
-> uniform num_video_frames sampling
-> dynamic 392x392 spatial tiles bounded by max_tiles_video
-> separate thumbnail frames
-> AutoGaze gazing_info
-> modified SigLIP features
-> NVILA projector and LLM generation
```

`hlvid_nvila_hd_subset.yaml` 기본값:

```yaml
num_video_frames: 128
num_video_frames_thumbnail: 64
max_tiles_video: 48
tile_size: 392x392
autogaze_chunk_size: 16
max_batch_size_autogaze: 16
max_batch_size_siglip: 32
```

## 3. Load Run Report

실행 후 `logs/run_report.json`을 읽어서 latency, memory, token consumption을 봅니다.

In [ ]:
from pathlib import Path
import json

RUN_DIR = Path('outputs/hlvid_nvila_real')
report_path = RUN_DIR / 'logs' / 'run_report.json'

if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report['metrics'], indent=2))
else:
    print(f'Missing report: {report_path}')

## 4. Latency Report

In [ ]:
if report_path.exists():
    print(json.dumps(report.get('latency_report', {}), indent=2))

`processor_latency_ms`는 NVILA processor 안에서 비디오 path를 받아 frame sampling, tile/thumbnail 생성, AutoGaze gazing_info, tokenizer 입력 구성을 수행하는 시간입니다.

`model_generate_latency_ms`는 NVILA `model.generate` 자체 시간입니다.

## 5. Memory Report

In [ ]:
if report_path.exists():
    print(json.dumps(report.get('memory_report', {}), indent=2))

`process_peak_rss_mib`는 프로세스 peak RSS이고, CUDA 사용 시 `cuda_max_memory_allocated_mib`가 같이 기록됩니다.

## 6. Token Consumption Report

In [ ]:
if report_path.exists():
    print(json.dumps(report.get('token_consumption_report', {}), indent=2))

`input_token_count`는 text token과 expanded video placeholder token을 포함한 `input_ids` 길이입니다.

`video_placeholder_token_count`는 NVILA가 visual feature를 꽂기 위해 확장한 video token 수입니다.

`output_new_token_count`는 새로 생성된 answer token 수입니다.

## 7. Per-Sample Predictions CSV

In [ ]:
import pandas as pd

csv_path = RUN_DIR / 'predictions' / 'hlvid_predictions.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display(df.head())
else:
    print(f'Missing CSV: {csv_path}')

## 8. What To Use

- HLVid 성능/재현: `scripts/evaluate_hlvid_nvila.py`
- 단일 비디오 시각화/debug: `scripts/infer_full.py`
- 공식 고해상도 처리 해석: `source_video -> official NVILA-HD processor -> AutoGaze -> SigLIP -> NVILA`